# Ensemble Training for Trained-Parameter Uncertainty

Train the same param set N times from perturbed initial conditions, look at the spread of
`best_params` across runs. Tight spread = loss actually constrains that parameter. Wide spread =
weakly constrained, don't trust a single run's point estimate.

Scope for now: shortwave-only, 15 params, the config already known to converge cleanly
(`trenberth_staged_phase1_sw.jl`, `batch_days=2`/`samples_per_batch=10`, relative-error weighting).

TODO, not done here: same idea for hyperparameter tuning (`batch_days`/`samples_per_batch` sweep) --
run a small ensemble per grid point instead of one realization, see section 7.

## 1. Setup

In [11]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", ".."))

using SpeedyCalibration, SpeedyWeather, SpeedyWeatherInternals.KernelLaunching
using Optimisers, Dates, Printf, Statistics, Random

  Activating project at `~/master_thesis/Code_SpeedyWeather/SpeedyCalibration.jl`


## 2. Perturbed Initial Condition

`RandomVorticity` replaces vorticity outright, meant to be the sole IC (e.g. Barotropic model), not
layered on top of `PrimitiveWetModel`'s default IC. `SmallVorticityPerturbation` below reuses its
noise kernel but *adds* to whatever vorticity the default IC already set, appended as an extra
component to `initial_conditions` (applied in sequence, same as SpeedyWeather's own convention).
`amplitude=1f-6`, ~100x smaller than `RandomVorticity`'s own default.

Verified separately: same seed -> identical perturbation, different seed -> different perturbation.

In [12]:
@kwdef mutable struct SmallVorticityPerturbation{NF} <: SpeedyWeather.AbstractInitialConditions
    "[OPTION] Power of the spectral distribution k^power (matches RandomVorticity's convention)"
    power::NF = -3

    "[OPTION] Perturbation amplitude [1/s] -- small vs. RandomVorticity's own default of 1f-4"
    amplitude::NF = 1.0f-6

    "[OPTION] Maximum wavenumber perturbed"
    max_wavenumber::Int = 20

    "[OPTION] Seed -- different seeds give different (reproducible) perturbations"
    seed::Int = 1
end
SmallVorticityPerturbation(SG::SpeedyWeather.SpectralGrid; kwargs...) =
    SmallVorticityPerturbation{SG.NF}(; kwargs...)

function SpeedyWeather.initialize!(
        vars::SpeedyWeather.Variables,
        ic::SmallVorticityPerturbation,
        model::SpeedyWeather.AbstractModel,
    )
    vor = vars.prognostic.vorticity
    NF  = real(eltype(vor))
    RNG = Random.Xoshiro(ic.seed)

    (; spectrum) = vor
    lmax   = spectrum.lmax + 1
    nlayers = size(vor, 2)
    power  = ic.power + 1
    (; amplitude, max_wavenumber) = ic

    nlm = SpeedyWeather.LowerTriangularArrays.nonzeros(spectrum)
    random_values_cpu_real = 2.0f0 .* rand(RNG, NF, nlm, nlayers) .- 1.0f0
    random_values_cpu_imag = 2.0f0 .* rand(RNG, NF, nlm, nlayers) .- 1.0f0im
    random_values_cpu = random_values_cpu_real .+ random_values_cpu_imag

    noise = similar(vor)[:, :, 1]
    random_values = SpeedyWeather.on_architecture(SpeedyWeather.architecture(noise), random_values_cpu)
    (; l_indices) = spectrum

    KernelLaunching.launch!(
        SpeedyWeather.architecture(noise), KernelLaunching.SpectralWorkOrder, size(noise),
        SpeedyWeather.random_vorticity_kernel!, noise, random_values, amplitude, power,
        l_indices, lmax, max_wavenumber,
    )

    base = vor[:, :, 1]
    perturbed = base .+ noise
    return SpeedyWeather.set!(vars, model; vorticity = perturbed, lf = 1)
end

println("SmallVorticityPerturbation defined.")

SmallVorticityPerturbation defined.


## 3. Modified Training Function

`calibrate!` builds its model internally, no hook for a custom IC. This is a copy with one line
changed (model construction, injects the perturbation). Everything else identical, same internal
calls. Will drift if `calibrate!` changes -- not a maintained duplicate.

In [13]:
function calibrate_ensemble_member!(
        param_specs  :: Vector{ParamSpec},
        optimizer,
        loss_config  :: LossConfig,
        config       :: TrainingConfig;
        ic_seed      :: Int,
        ic_amplitude :: Float32 = 1.0f-6,
        save_dir     :: Union{AbstractString,Nothing} = nothing,
    )
    n_params    = length(param_specs)
    param_names = [spec.name for spec in param_specs]
    flux_keys   = loss_config.flux_keys

    history = Dict{Symbol,Vector}(
        :batch => Int[], :loss => Float32[], :smoothed_loss => Float32[],
        :elapsed_time => Float64[], :param_change => Float32[], :lr => Float32[],
    )
    for k in flux_keys; history[k] = Float32[]; end
    for name in param_names
        history[name] = Float32[]
        history[Symbol("grad_", name)] = Float32[]
        history[Symbol("gradstd_", name)] = Float32[]
    end
    loss_window = Float32[]

    # Build model -- ONLY CHANGE vs. calibrate!: perturbed initial_conditions
    sg     = SpectralGrid(trunc=config.trunc, nlayers=config.nlayers)
    planet = Earth(sg; daily_cycle=config.daily_cycle, seasonal_cycle=false)
    ic_base = InitialConditions(sg, PrimitiveWet)
    perturbed_ic = (; ic_base..., perturbation = SmallVorticityPerturbation(sg; seed=ic_seed, amplitude=ic_amplitude))
    model  = PrimitiveWetModel(sg; planet=planet, initial_conditions=perturbed_ic)
    p      = vec(parameters(model))

    init_phys = Float32[]
    for spec in param_specs
        val = isnothing(spec.initial) ? Float32(SpeedyCalibration.get_by_path(p, spec.path)) : spec.initial
        SpeedyCalibration.set_by_path!(p, spec.path, val)
        push!(init_phys, val)
    end
    model = SpeedyWeather.reconstruct(model, p)
    if config.dt !== nothing
        SpeedyWeather.set!(model.time_stepping; Δt=config.dt)
    end
    sim = initialize!(model)
    sim.variables.prognostic.clock.time = config.start_date
    SpeedyWeather.initialize!(sim; period=Day(365*100), output=false)

    clock            = sim.variables.prognostic.clock
    steps_per_day    = ceil(Int, Millisecond(Day(1)).value / Millisecond(clock.Δt).value)
    batch_steps      = ceil(Int, config.batch_days * steps_per_day)
    steps_per_sample = max(1, batch_steps ÷ config.samples_per_batch)

    opt_params  = Float32[SpeedyCalibration.to_raw(init_phys[i], spec.bounds[1], spec.bounds[2])
                          for (i, spec) in enumerate(param_specs)]
    phys_values = Float32[SpeedyCalibration.sigmoid_param(opt_params[i], spec.bounds[1], spec.bounds[2])
                          for (i, spec) in enumerate(param_specs)]

    opt_state  = Optimisers.setup(optimizer, opt_params)
    current_lr = optimizer isa Optimisers.Adam ? Float32(optimizer.eta) : NaN32

    best_smoothed_loss = Inf32
    best_phys_values   = copy(phys_values)
    best_batch         = 0
    batches_since_best = 0
    lr_decay_count     = 0
    prev_phys          = copy(phys_values)
    stop_reason        = "max batches reached"
    converged          = false
    start_time         = time()

    log_file = save_dir !== nothing ? (mkpath(save_dir);
                                       open(joinpath(save_dir, "training.log"), "w")) : nothing
    io = SpeedyCalibration._output_io(config.verbose, log_file)

    SpeedyCalibration._print_header(io, param_specs, phys_values, opt_params, loss_config, config, current_lr)
    @printf(io, "\nPerturbed IC: seed=%d, amplitude=%.1e\n", ic_seed, ic_amplitude)

    @printf(io, "\nSpinup (%d days)...\n", config.spinup_days)
    t_spinup = time()
    for _ in 1:(config.spinup_days * steps_per_day)
        SpeedyWeather.timestep!(sim)
    end
    @printf(io, "Spinup complete in %.1f s.\n\n", time() - t_spinup)
    flush(io)

    if config.warmup_enzyme
        enzyme_warmup(sim, loss_config, param_specs)
    end

    println(io, "Starting training...")
    println(io, "-" ^ 70)

    for batch in 1:config.max_batches
        all_grads  = [Float32[] for _ in 1:n_params]
        flux_accum = Dict{Symbol,Vector{Float32}}(k => Float32[] for k in flux_keys)

        for _ in 1:config.samples_per_batch
            for _ in 1:steps_per_sample
                SpeedyWeather.timestep!(sim)
            end
            grads, means, _ = SpeedyCalibration.compute_gradients!(
                sim.variables, sim.model, loss_config, param_specs)

            if all(isfinite, grads) && all(isfinite(means[k]) for k in flux_keys)
                for (i, g) in enumerate(grads); push!(all_grads[i], g); end
                for k in flux_keys; push!(flux_accum[k], means[k]); end
            end
        end

        remainder = batch_steps - config.samples_per_batch * steps_per_sample
        for _ in 1:remainder; SpeedyWeather.timestep!(sim); end

        if isempty(flux_accum[flux_keys[1]])
            println(io, "Batch $batch: all gradient samples invalid, stopping.")
            stop_reason = "all gradient samples invalid"
            break
        end

        raw_mean_grads = Float32[mean(g) for g in all_grads]
        raw_std_grads  = Float32[length(g) > 1 ? std(g; corrected=false) : 0f0 for g in all_grads]

        for i in 1:n_params
            raw_mean_grads[i] *= param_specs[i].grad_scale *
                SpeedyCalibration.sigmoid_grad_factor(opt_params[i], param_specs[i].bounds[1], param_specs[i].bounds[2])
        end

        mean_fluxes = Dict{Symbol,Float32}(k => mean(flux_accum[k]) for k in flux_keys)
        loss        = SpeedyCalibration.compute_loss(mean_fluxes, loss_config)
        elapsed     = time() - start_time

        scaled_grads = copy(raw_mean_grads)
        grad_norm = sqrt(sum(scaled_grads .^ 2))
        if grad_norm > config.grad_clip
            scaled_grads .*= config.grad_clip / grad_norm
        end

        push!(loss_window, loss)
        length(loss_window) > config.loss_window_size && popfirst!(loss_window)
        smoothed_loss = mean(loss_window)

        param_change = mean(abs.(phys_values .- prev_phys) ./ max.(abs.(phys_values), 1f-6))

        if smoothed_loss < best_smoothed_loss
            best_smoothed_loss = smoothed_loss
            best_phys_values   = copy(phys_values)
            best_batch         = batch
            batches_since_best = 0
        else
            batches_since_best += 1
        end

        if config.enable_lr_decay &&
                batches_since_best >= config.lr_plateau_patience &&
                current_lr > config.min_lr &&
                lr_decay_count < config.max_lr_decays
            old_lr     = current_lr
            current_lr = max(current_lr * config.lr_decay_factor, config.min_lr)
            opt_state  = Optimisers.setup(Optimisers.Adam(current_lr), opt_params)
            batches_since_best = 0
            lr_decay_count += 1
            @printf(io, "  ↓ LR: %.2e → %.2e (decay #%d)\n", old_lr, current_lr, lr_decay_count)
        end

        prev_phys = copy(phys_values)
        opt_state, opt_params = Optimisers.update(opt_state, opt_params, scaled_grads)
        phys_values = Float32[SpeedyCalibration.sigmoid_param(opt_params[i], spec.bounds[1], spec.bounds[2])
                               for (i, spec) in enumerate(param_specs)]

        _variables = sim.variables
        new_p = vec(parameters(sim.model))
        for (i, spec) in enumerate(param_specs)
            SpeedyCalibration.set_by_path!(new_p, spec.path, phys_values[i])
        end
        updated_model = SpeedyWeather.reconstruct(sim.model, new_p)
        updated_model.time_stepping.first_step_euler = false
        sim = Simulation(_variables, updated_model)

        push!(history[:batch], batch)
        push!(history[:loss], loss)
        push!(history[:smoothed_loss], smoothed_loss)
        push!(history[:elapsed_time], elapsed)
        push!(history[:param_change], param_change)
        push!(history[:lr], current_lr)
        for k in flux_keys; push!(history[k], mean_fluxes[k]); end
        for (i, name) in enumerate(param_names)
            push!(history[name], phys_values[i])
            push!(history[Symbol("grad_", name)], raw_mean_grads[i])
            push!(history[Symbol("gradstd_", name)], raw_std_grads[i])
        end

        if batch <= 10 || batch % 5 == 0
            flux_str = join([@sprintf("%s=%5.1f", k, mean_fluxes[k]) for k in flux_keys], " ")
            @printf(io, "Batch %3d | LR %.1e | %s | L̄ %8.2f | Δp %.1e\n",
                    batch, current_lr, flux_str, smoothed_loss, param_change)
            flush(io)
        end

        if smoothed_loss < config.loss_threshold
            converged   = true
            stop_reason = "smoothed loss below threshold ($(config.loss_threshold))"
            @printf(io, "CONVERGED: smoothed_loss %.4f < %.4f\n", smoothed_loss, config.loss_threshold)
            break
        end

        lr_decay_exhausted = !config.enable_lr_decay || lr_decay_count >= config.max_lr_decays
        if lr_decay_exhausted && batches_since_best >= config.patience
            stop_reason = "no improvement for $(config.patience) batches" *
                          (config.enable_lr_decay ? " after exhausting $(config.max_lr_decays) LR decays" : "")
            @printf(io, "EARLY STOP: %s (best batch %d, best smoothed_loss %.4f)\n",
                    stop_reason, best_batch, best_smoothed_loss)
            break
        end
    end

    final_params = Dict(spec.name => phys_values[i] for (i, spec) in enumerate(param_specs))
    best_params  = Dict(spec.name => best_phys_values[i] for (i, spec) in enumerate(param_specs))
    conv_info = (
        converged = converged, stop_reason = stop_reason,
        total_batches = length(history[:batch]), best_smoothed_loss = best_smoothed_loss,
        best_batch = best_batch, total_time = time() - start_time,
    )

    SpeedyCalibration._print_summary(io, param_specs, init_phys, phys_values, history, flux_keys, loss_config, conv_info)

    result = TrainingResult(history, final_params, best_params, conv_info, config, loss_config, param_specs)

    if save_dir !== nothing
        log_file !== nothing && close(log_file)
        save_artifacts(result, save_dir)
    end

    return result
end

println("calibrate_ensemble_member! defined.")

calibrate_ensemble_member! defined.


## 4. Shortwave-Only Parameter Set

Same 15 params as `trenberth_staged_phase1_sw.jl`, current relative-error loss weighting.

In [14]:
param_specs = [
    ParamSpec(:cloud_albedo, [:shortwave_radiation, :clouds, :cloud_albedo];
        bounds=(0.25f0, 0.95f0), initial=0.60f0),
    ParamSpec(:stratocumulus_cover_max, [:shortwave_radiation, :clouds, :stratocumulus_cover_max];
        bounds=(0.25f0, 0.95f0), initial=0.60f0),
    ParamSpec(:stratocumulus_albedo, [:shortwave_radiation, :clouds, :stratocumulus_albedo];
        bounds=(0.10f0, 0.90f0), initial=0.50f0),
    ParamSpec(:precipitation_weight, [:shortwave_radiation, :clouds, :precipitation_weight];
        bounds=(0.0f0, 0.8f0), initial=0.20f0),
    ParamSpec(:absorptivity_water_vapor, [:shortwave_radiation, :transmissivity, :absorptivity_water_vapor];
        bounds=(60f0, 140f0), initial=75f0, grad_scale=0.01f0),
    ParamSpec(:absorptivity_dry_air, [:shortwave_radiation, :transmissivity, :absorptivity_dry_air];
        bounds=(0.005f0, 0.060f0), initial=0.03135f0),
    ParamSpec(:absorptivity_aerosol, [:shortwave_radiation, :transmissivity, :absorptivity_aerosol];
        bounds=(0.005f0, 0.060f0), initial=0.03135f0),
    ParamSpec(:ozone_absorption, [:shortwave_radiation, :radiative_transfer, :ozone_absorption];
        bounds=(0.002f0, 0.020f0), initial=0.01f0),
    ParamSpec(:albedo_land, [:albedo, :land, :albedo_land];
        bounds=(0.10f0, 0.70f0), initial=0.40f0),
    ParamSpec(:albedo_high_vegetation, [:albedo, :land, :albedo_high_vegetation];
        bounds=(0.04f0, 0.26f0), initial=0.15f0),
    ParamSpec(:albedo_low_vegetation, [:albedo, :land, :albedo_low_vegetation];
        bounds=(0.05f0, 0.35f0), initial=0.20f0),
    ParamSpec(:albedo_snow, [:albedo, :land, :albedo_snow];
        bounds=(0.15f0, 0.75f0), initial=0.40f0),
    ParamSpec(:snow_depth_scale, [:albedo, :land, :snow_depth_scale];
        bounds=(0.005f0, 0.20f0), initial=0.05f0),
    ParamSpec(:albedo_ocean, [:albedo, :ocean, :albedo_ocean];
        bounds=(0.02f0, 0.10f0), initial=0.06f0),
    ParamSpec(:albedo_ice, [:albedo, :ocean, :albedo_ice];
        bounds=(0.30f0, 0.90f0), initial=0.60f0),
]

relerror_loss = LossConfig(
    [:osr, :sru, :srd, :olr, :lrd, :lru];
    targets = Dict(:osr => 101.9f0, :sru =>  23.0f0, :srd => 168.0f0,
                   :olr => 235.0f0, :lrd => 333.0f0, :lru => 398.0f0),
    weights = Dict(:osr => 1.00000f0, :sru => 19.62875f0, :srd => 0.36790f0,
                   :olr => 0.18802f0, :lrd => 0.09364f0,  :lru => 0.06555f0),
)

println("$(length(param_specs)) SW-only trainable parameters, relative-error loss weighting.")

15 SW-only trainable parameters, relative-error loss weighting.


## 5. Ensemble Loop

N=5, resume-safe per member. Each member is a full training run (~1h) -- multi-hour total.

In [15]:
const N_ENSEMBLE = 20
const ENSEMBLE_DIR = joinpath(@__DIR__, "..", "output", "trenberth_ensemble_uncertainty", "sw_only")
mkpath(ENSEMBLE_DIR)

members = TrainingResult[]

for member in 1:N_ENSEMBLE
    println("\n", "=" ^ 70)
    println("ENSEMBLE MEMBER ", member, " / ", N_ENSEMBLE, "  (ic_seed=", member, ")")
    println("=" ^ 70)

    member_dir  = joinpath(ENSEMBLE_DIR, "member_$member")
    save_path   = joinpath(member_dir, "result.jld2")

    if isfile(save_path)
        result = load_result(save_path)
        println("Loaded existing result for member ", member, "  best_batch=", result.conv_info.best_batch)
    else
        result = calibrate_ensemble_member!(
            param_specs,
            Optimisers.Adam(5f-3),
            relerror_loss,
            TrainingConfig(
                spinup_days       = 180,
                batch_days        = 2.0,
                samples_per_batch = 10,
                max_batches       = 300,
                grad_clip         = 5f0,
                trunc             = 31,
                nlayers           = 8,
                loss_threshold    = 1f-6,
                enable_lr_decay   = false,
                daily_cycle       = true,
            ),
            ic_seed  = member,
            save_dir = member_dir,
        )
    end

    push!(members, result)
    @printf("Member %d: best_batch=%d  best_smoothed_loss=%.2f\n",
            member, result.conv_info.best_batch, result.conv_info.best_smoothed_loss)
end

println("\nAll ", N_ENSEMBLE, " ensemble members complete.")


ENSEMBLE MEMBER 1 / 20  (ic_seed=1)
Loaded existing result for member 

1  best_batch=299
Member 1: best_batch=299  best_smoothed_loss=85.77

ENSEMBLE MEMBER 2 / 20  (ic_seed=2)
Loaded existing result for member 2  best_batch=296
Member 2: best_batch=296  best_smoothed_loss=87.00

ENSEMBLE MEMBER 3 / 20  (ic_seed=3)
Loaded existing result for member 3  best_batch=284
Member 3: best_batch=284  best_smoothed_loss=84.76

ENSEMBLE MEMBER 4 / 20  (ic_seed=4)
Loaded existing result for member 4  best_batch=278
Member 4: best_batch=278  best_smoothed_loss=88.43

ENSEMBLE MEMBER 5 / 20  (ic_seed=5)
Loaded existing result for member 5  best_batch=294
Member 5: best_batch=294  best_smoothed_loss=86.14

ENSEMBLE MEMBER 6 / 20  (ic_seed=6)
SpeedyCalibration.jl: calibrate!
   1. cloud_albedo                     = 0.6000  [0.250, 0.950]  raw₀=0.000
   2. stratocumulus_cover_max          = 0.6000  [0.250, 0.950]  raw₀=0.000
   3. stratocumulus_albedo             = 0.5000  [0.100, 0.900]  raw₀=0.000
   4. precipitation_weight             = 0.2000  [0.000, 0.800]  raw₀=-1.

   0%  ETA: 3:58:19 (2000-09-16, 601.26 years/day, 112 m/s, [ -86,   27] ˚C)

Spinup complete in 70.9 s.

Warming up Enzyme (compiling AD rules on actual model)...
Enzyme warmup complete in 0.2 s.


   0%  ETA: 3:59:07 (2000-09-17, 599.22 years/day, 113 m/s, [ -87,   25] ˚C)

...

Batch 290 | LR 5.0e-03 | osr=102.2 sru= 22.4 srd=169.5 olr=255.3 lrd=330.3 lru=393.0 | L̄    85.21 | Δp 1.8e-03


   2%  ETA: 11:08:51 (2002-04-29, 210.75 years/day, 102 m/s, [-104,   41] ˚C)

Batch 295 | LR 5.0e-03 | osr=101.7 sru= 23.0 srd=171.2 olr=255.0 lrd=329.2 lru=391.1 | L̄    84.70 | Δp 1.1e-03


   2%  ETA: 11:10:15 (2002-05-09, 210.26 years/day, 114 m/s, [-103,   42] ˚C)

Batch 300 | LR 5.0e-03 | osr=101.7 sru= 22.7 srd=170.4 olr=254.8 lrd=329.2 lru=392.1 | L̄    84.00 | Δp 1.5e-03

TRAINING SUMMARY
  cloud_albedo                     0.6000 → 0.8035
  stratocumulus_cover_max          0.6000 → 0.6839
  stratocumulus_albedo             0.5000 → 0.5850
  precipitation_weight             0.2000 → 0.5004
  absorptivity_water_vapor         75.0000 → 89.8423
  absorptivity_dry_air             0.0314 → 0.0302
  absorptivity_aerosol             0.0314 → 0.0386
  ozone_absorption                 0.0100 → 0.0083
  albedo_land                      0.4000 → 0.2116
  albedo_high_vegetation           0.1500 → 0.1048
  albedo_low_vegetation            0.2000 → 0.1163
  albedo_snow                      0.4000 → 0.6217
  snow_depth_scale                 0.0500 → 0.0333
  albedo_ocean                     0.0600 → 0.0916
  albedo_ice                       0.6000 → 0.8413
----------------------------------------------------------------------
  Final osr : 101.73 W/m²  (targ

Filter out any member whose training broke before completing a single batch
(`best_batch=0`, `history` empty -- `calibrate_ensemble_member!` breaks immediately if every
gradient sample in batch 1 is non-finite). Its `best_params` would just be the untrained initial
values, not a real fit -- including it would corrupt the mean/std below, not just look odd in a
table. Sections 6-8 use `valid_members` from here on, not the raw `members` list.

In [20]:
valid_members  = [r for r in members if r.conv_info.best_batch > 0]
failed_indices = [i for (i, r) in enumerate(members) if r.conv_info.best_batch == 0]

println(length(valid_members), " / ", length(members), " members trained successfully.")
if !isempty(failed_indices)
    println("Excluded (best_batch=0, no history -- all gradient samples invalid on batch 1): ",
            join(["member_$i" for i in failed_indices], ", "))
end

19 / 20 members trained successfully.


## 6. Parameter Uncertainty

Mean +/- std of `best_params` across the N members.

In [22]:
println(@sprintf("%-28s  %10s  %10s  %10s  %8s", "param", "mean", "std", "std/mean", "initial"))
println("-" ^ 72)
for spec in param_specs
    vals = [m.best_params[spec.name] for m in valid_members]
    μ, σ = mean(vals), std(vals)
    rel  = μ != 0 ? abs(σ / μ) : NaN
    @printf("%-28s  %10.4f  %10.4f  %10.2f%%  %8.4f\n", spec.name, μ, σ, 100*rel, isnothing(spec.initial) ? NaN32 : spec.initial)
end

param                               mean         std    std/mean   initial
------------------------------------------------------------------------
cloud_albedo                      0.8024      0.0035        0.43%    0.6000
stratocumulus_cover_max           0.7228      0.0183        2.54%    0.6000
stratocumulus_albedo              0.6316      0.0224        3.54%    0.5000
precipitation_weight              0.4985      0.0080        1.60%    0.2000
absorptivity_water_vapor         89.4176      0.5281        0.59%   75.0000
absorptivity_dry_air              0.0306      0.0008        2.52%    0.0314
absorptivity_aerosol              0.0389      0.0004        0.99%    0.0314
ozone_absorption                  0.0085      0.0003        3.25%    0.0100
albedo_land                       0.2186      0.0056        2.56%    0.4000
albedo_high_vegetation            0.1038      0.0024        2.27%    0.1500
albedo_low_vegetation             0.1205      0.0033        2.76%    0.2000
albedo_snow     

## 7. Convergence Check

Before trusting the parameter spread above as real uncertainty: did all N members actually
converge to comparably good fits, or is some of that spread just "some runs fit worse"? Loss and
training-batch flux values at each member's own `best_batch`. Training-batch numbers are a proxy,
not true equilibrium (see section 8) -- this is only a cheap first sanity check.

In [23]:
println(@sprintf("%-10s  %10s  %10s  %8s %8s %8s %8s %8s %8s",
        "member", "best_batch", "best_loss", "osr", "sru", "srd", "olr", "lrd", "lru"))
println("-" ^ 92)
for (i, r) in enumerate(valid_members)
    bb = r.conv_info.best_batch
    h  = r.history
    idx = findfirst(==(bb), h[:batch])
    fluxvals = [h[k][idx] for k in [:osr, :sru, :srd, :olr, :lrd, :lru]]
    @printf("%-10s  %10d  %10.2f  %8.2f %8.2f %8.2f %8.2f %8.2f %8.2f\n",
            "member_$i", bb, r.conv_info.best_smoothed_loss, fluxvals...)
end

println()
println("stop reasons:")
for (i, r) in enumerate(valid_members)
    println("  member_$i: ", r.conv_info.stop_reason, "  (total_batches=", r.conv_info.total_batches, ")")
end

member      best_batch   best_loss       osr      sru      srd      olr      lrd      lru
--------------------------------------------------------------------------------------------
member_1           299       85.77    102.12    22.63   170.47   254.85   329.45   391.35
member_2           296       87.00    100.95    22.48   170.26   255.25   330.08   391.96
member_3           284       84.76    100.89    22.59   170.41   255.21   329.56   391.46
member_4           278       88.43    101.55    22.22   169.90   255.24   329.81   392.57
member_5           294       86.14    101.04    22.44   170.66   255.22   330.45   393.20
member_6           268       88.93    100.70    22.52   170.58   255.15   330.83   394.65
member_7           300       82.66    101.51    22.62   170.21   254.87   329.44   391.41
member_8           300       83.96    101.19    22.57   170.51   254.55   330.52   393.20
member_9           293       84.03    100.74    22.68   171.18   255.40   330.80   393.01
member_

## 8. Per-Member Flux Bias at True Equilibrium

The real version of section 7 -- training-batch numbers aren't trustworthy on their own (this
project has been burned by training-metric/equilibrium mismatches before, see
`project_trenberth_lw_transmissivity_gradscale_fix` memory). Runs `run_climate_validation`
(`n_years=7`/`stat_years=5`, the project standard, not a screening budget) once per member.

**Expensive: N x ~50min, not cached.** Don't run this casually alongside other background work.

In [ ]:
equilibrium_biases = Dict{Symbol,Vector{Float32}}(k => Float32[] for k in [:osr, :sru, :srd, :olr, :lrd, :lru])

println(@sprintf("%-10s  %8s %8s %8s %8s %8s %8s", "member", "osr", "sru", "srd", "olr", "lrd", "lru"))
println("-" ^ 68)
for (i, r) in enumerate(valid_members)
    clm = run_climate_validation(r; n_years=7, stat_years=5, dt=Minute(20))
    biasvals = Float32[]
    for k in [:osr, :sru, :srd, :olr, :lrd, :lru]
        tgt = r.loss_config.targets[k]
        b = getproperty(clm.trained, k) - tgt
        push!(equilibrium_biases[k], b)
        push!(biasvals, b)
    end
    @printf("%-10s  %+8.2f %+8.2f %+8.2f %+8.2f %+8.2f %+8.2f\n", "member_$i", biasvals...)
end

println()
println("mean +/- std across members:")
for k in [:osr, :sru, :srd, :olr, :lrd, :lru]
    vals = equilibrium_biases[k]
    @printf("  %-4s  %+7.2f +/- %5.2f\n", k, mean(vals), std(vals))
end

member           osr      sru      srd      olr      lrd      lru
--------------------------------------------------------------------
Climate run: default ...


   4%  ETA: 0:32:38 (2000-07-02, 296.30 years/day,  96 m/s, [ -84,   29] ˚C)

LoadError: InterruptException:

## 9. TODO: same approach for hyperparameter tuning

Not implemented here. Extend to `batch_days`/`samples_per_batch` (`trenberth_batchdays_gradcount_sweep/`):
run a small ensemble (e.g. N=3) per grid point, report mean +/- std of the equilibrium bias instead
of one number. Would tell us whether a config difference is real or within the model's own noise --
currently assumed, never checked. Deferred until this notebook's single-config version is validated
and its cost is known.